# Phase 4 — Observed mobility and econometric modeling
## Sources and methods
2024 ACS and Google Maps distances from merged_2024.csv; population from ztca.zip; Dewey 2024 all-period reported device origins; 2024 Georgia CBG and 2026 ZCTA TIGER boundaries. Area intersection in EPSG:5070 approximates allocation, not population weights. Boundary vintage consistency and provider reporting rules remain unverified. Unreported origins are not zeros. Models are descriptive conditional associations, with 23 origin clusters; asymptotic p-values should be interpreted cautiously.

In [ ]:
import pandas as pd, numpy as np, geopandas as g, zipfile, statsmodels.formula.api as smf, statsmodels.api as sm
from pathlib import Path
import matplotlib.pyplot as plt
D=Path('/content') if Path('/content').exists() else Path('/mnt/data'); O=D/'Phase4_results'; O.mkdir(exist_ok=True)
x=pd.read_csv(D/'merged_2024.csv',dtype={'ZCTA':str})
with zipfile.ZipFile(D/'ztca.zip') as z: p=pd.read_csv(z.open('ACSDT5Y2024.B01003-Data.csv'))
p['ZCTA']=p.GEO_ID.str.extract(r'860Z200US(\d{5})'); p['population']=pd.to_numeric(p.B01003_001E,errors='coerce'); x=x.merge(p[['ZCTA','population']].dropna(),on='ZCTA',validate='many_to_one')
b=g.read_file(D/'tl_2024_13_bg (1).zip',columns=['GEOID']).to_crs(5070)
# Read national file selectively by attribute using GDAL SQL filter.
z=g.read_file(D/'tl_2026_us_zcta520.zip',where='ZCTA5CE20 IN ('+','.join("'"+i+"'" for i in x.ZCTA.unique())+')',columns=['ZCTA5CE20']).to_crs(5070).rename(columns={'ZCTA5CE20':'ZCTA'})
assert len(z)==24
b['geometry']=b.geometry.make_valid(); z['geometry']=z.geometry.make_valid(); b['area']=b.area
w=g.overlay(b,z,how='intersection',keep_geom_type=True); w['weight']=w.geometry.area/w['area']; w=w[['GEOID','ZCTA','weight']]; w=w[w.weight>1e-9]; w.to_csv(O/'crosswalk.csv',index=False)
with zipfile.ZipFile(D/'Atlanta_Foot_Traffic_2024.zip') as f: v=pd.read_csv(f.open('Origin_Destination_Flows.csv'),dtype={'Origin_CBG':str,'Destination_CBG':str})
v=v[v.Period=='All']; a=v.merge(w,left_on='Origin_CBG',right_on='GEOID'); a['allocated']=a.Device_Count*a.weight
q=a.groupby(['ZCTA','Mall','Month']).allocated.sum().rename('flow').reset_index().rename(columns={'Mall':'Destination'}); d=x.merge(q,on=['ZCTA','Destination']); d['log_income']=np.log(d.income); d['log_distance']=np.log1p(d.driving_miles); d['log_pop']=np.log(d.population); d['luxury']=(d.Destination!='Ponce City Market').astype(int); d.to_csv(O/'panel.csv',index=False)
print('crosswalk',len(w),'CBGs',w.GEOID.nunique(),'max weight sum',w.groupby('GEOID').weight.sum().max(),'panel',len(d), 'income complete',d.income.notna().sum()); print(d.groupby('Destination').flow.sum())
c=d.dropna(subset=['log_income','log_pop','log_distance']).copy(); c['income_luxury']=c.log_income*c.luxury
models={'gravity':'flow ~ log_distance + log_income + log_pop + C(Destination) + C(Month)','sorting':'flow ~ log_distance + income_luxury + C(ZCTA) + C(Destination) + C(Month)','controls':'flow ~ log_distance + log_income + log_pop + gini + education_share + poverty_rate + np.log(home_value) + C(Destination) + C(Month)'}
rows=[]
for name,formula in models.items():
 m=smf.glm(formula,c,family=sm.families.Poisson()).fit(cov_type='cluster',cov_kwds={'groups':c.ZCTA}); ci=m.conf_int(); print(name,m.converged,len(c));
 for k in ['log_distance','log_income','log_pop','income_luxury','gini','education_share','poverty_rate','np.log(home_value)']:
  if k in m.params: rows.append([name,k,m.params[k],m.bse[k],m.pvalues[k],*ci.loc[k],len(c),c.ZCTA.nunique()]); print(k,m.params[k],m.pvalues[k])
r=pd.DataFrame(rows,columns=['model','term','coefficient','se','p','lower','upper','n','clusters']);r.to_csv(O/'coefficients.csv',index=False)
c['weighted_income']=c.flow*c.income; profile=c.groupby('Destination').agg(flow=('flow','sum'),weighted=('weighted_income','sum')); profile['weighted_income']=profile.weighted/profile.flow; print(profile); profile.to_csv(O/'income_profile.csv'); profile.weighted_income.plot.bar(ylabel='Weighted origin median household income ($)',rot=15);plt.tight_layout();plt.savefig(O/'income_profile.png');plt.close()
rr=r[r.term.isin(['log_distance','log_income','income_luxury'])]; plt.errorbar(rr.coefficient,range(len(rr)),xerr=[rr.coefficient-rr.lower,rr.upper-rr.coefficient],fmt='o');plt.yticks(range(len(rr)),rr.model+' / '+rr.term);plt.axvline(0);plt.tight_layout();plt.savefig(O/'coefficients.png');plt.close()


## Findings
Distance is negatively associated with flow in all three specifications. Income is positive but not significant at 5% in the baseline, and weaker after controls. Income interacted with the Buckhead destination indicator is positive but imprecise. The weighted income profile is descriptive and refers to origin neighborhoods, not individuals. Monthly counts are not annual unique visitors. Only 23 income-complete ZCTAs are analyzed.